# Sargam ML - Colab Training Pipeline

Train gamaka, timbre, and onset models on Saraga dataset.
Models auto-pushed to Hugging Face Hub for HF Spaces deployment.

In [ ]:
# @title 1. Setup & Authentication
import os, sys, subprocess, json, getpass

# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

# Hugging Face token (for pushing models)
HF_TOKEN = getpass.getpass('Enter HF_TOKEN (write access): ')
HF_USERNAME = input('HF Username: ')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_USERNAME'] = HF_USERNAME

# Project directory in Drive (persists across sessions)
PROJECT_DIR = '/content/drive/MyDrive/sargam_ml'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

# Clone repo if not exists
if not os.path.exists('sargam_ML'):
    !git clone https://github.com/YOUR_USERNAME/sargam_ML.git
%cd sargam_ML

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q huggingface_hub[cli]

print('✅ Setup complete')

In [ ]:
# @title 2. Download Saraga Dataset
# Choose: 'mini' (210 MB, fast) or 'full' (14 GB, slow)
DATASET_MODE = 'mini'  # @param ['mini', 'full']

import yaml
config_path = 'config/dataset_config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

if DATASET_MODE == 'mini':
    config['download']['record_id'] = '16411600'  # Saraga Mini
    print('📥 Downloading Saraga Mini (~210 MB)...')
else:
    config['download']['record_id'] = '4301737'   # Saraga 1.5 Full
    print('📥 Downloading Saraga Full (~14 GB)...')

with open(config_path, 'w') as f:
    yaml.dump(config, f)

# Run download
!python data/download_saraga.py

# Prepare train/val/test splits
!python data/prepare_splits.py

print('✅ Dataset ready')

In [ ]:
# @title 3. Train Models
import os
os.makedirs('models/gamaka', exist_ok=True)
os.makedirs('models/timbre', exist_ok=True)
os.makedirs('models/onset', exist_ok=True)

# Train gamaka classifier (RF on contour features)
print('🎯 Training gamaka classifier...')
!python scripts/train_gamaka.py

# Train timbre classifier (RF on MFCC+spectral)
print('🎵 Training timbre classifier...')
!python scripts/train_timbre.py

# Train onset detector (optional, needs annotations)
print('🎵 Training onset detector...')
!python scripts/train_onset.py 2>/dev/null || echo 'Skipped (no onset annotations)'

print('✅ Training complete')

In [ ]:
# @title 4. Push Models to Hugging Face Hub
from huggingface_hub import HfApi, create_repo

repo_id = f"{HF_USERNAME}/sargam-models"

# Create repo if not exists
try:
    create_repo(repo_id, private=True, token=HF_TOKEN)
    print(f'Created repo: {repo_id}')
except:
    print(f'Repo exists: {repo_id}')

api = HfApi(token=HF_TOKEN)

# Upload all model files
for root, dirs, files in os.walk('models'):
    for f in files:
        local_path = os.path.join(root, f)
        repo_path = os.path.relpath(local_path, 'models')
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=repo_id,
            repo_type='model'
        )
        print(f'  Uploaded: {repo_path}')

print(f'✅ Models pushed to https://huggingface.co/{repo_id}')

In [ ]:
# @title 5. Quick Validation
import numpy as np
import soundfile as sf
import tempfile

# Generate test audio
sr = 16000
t = np.linspace(0, 2, 2*sr)
audio = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)

with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
    sf.write(f.name, audio, sr)
    test_path = f.name

# Test pipeline loads from HF Hub
from src.pipeline.orchestrator import create_pipeline
pipeline = create_pipeline()
result = pipeline.run(test_path, tonic_hz=440.0)

print('Pipeline test:')
print(f"  Success: {result['success']}")
if result['success']:
    d = result['data']
    print(f"  Tonic: {d['tonic']} ({d['tonic_hz']:.1f} Hz)")
    print(f"  Tempo: {d['tempo']} ({d['tempo_bpm']:.0f} BPM)")
    print(f"  Instrument: {d['instrument']}")
    print(f"  Notes: {len(d['notes'])}")
    for n in d['notes'][:3]:
        print(f"    {n['swara']}{n['variant']} {n['octave']} @ {n['onset_sec']:.2f}s")

os.unlink(test_path)
print('✅ Validation passed')

In [ ]:
# @title 6. Update HF Space to Use Hub Models
print("""
In your HF Space, update the model loading to fetch from Hub:

```python
from huggingface_hub import hf_hub_download

# In src/models/gamaka_classifier.py __init__:
model_path = hf_hub_download(
    repo_id='{HF_USERNAME}/sargam-models',
    filename='gamaka/gamaka_classifier.joblib',
    token=os.getenv('HF_TOKEN')
)
```

Or set HF_HOME=/data/models in Space secrets for auto-cache.
""".format(HF_USERNAME=HF_USERNAME))